In [101]:
import pandas as pd

df = pd.read_csv('cadastro_clientes.csv')

def nao_e_numero(val):
    if pd.isna(val):
        return False
    text = str(val).strip().upper()
    # remove prefixos e formatação brasileira/americana comuns
    cleaned = text.replace("R$", "").replace(" ", "").replace(".", "").replace(",", ".").strip()
    try:
        float(cleaned)
        return False
    except ValueError:
        return True

outliers = df[df["renda_anual"].apply(nao_e_numero)]["renda_anual"]
print("Qtd de outliers:", len(outliers))
print(outliers.value_counts().head(50))

In [103]:
import pandas as pd
import re
from dateutil import parser as dateutil_parser
from datetime import date

# ── Valores que indicam ausência de informação ─────────────────────────────────
INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}


def parse_date(raw) -> str | None:
    """
    Tenta interpretar qualquer string de data e retorna no formato dd/mm/yyyy.
    Retorna None se o valor for incoerente / ausente.
    """
    if pd.isna(raw):
        return None

    text = str(raw).strip()

    if text.lower() in INVALID_VALUES:
        return None

    # Formato textual: "11 Jun 1984"
    month_abbr = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
    if re.fullmatch(rf"\d{{1,2}} {month_abbr} \d{{4}}", text, re.IGNORECASE):
        try:
            dt = dateutil_parser.parse(text, dayfirst=True)
            return dt.strftime("%d/%m/%Y")
        except Exception:
            return None

    # Formatos numéricos
    parts = re.split(r"[-/.]", text)
    if len(parts) != 3:
        return None

    try:
        a, b, c = parts

        if len(a) == 4:
            year, p2, p3 = int(a), int(b), int(c)
            month, day = (p2, p3) if 1 <= p2 <= 12 else (p3, p2)

        elif len(c) == 4:
            year, p1, p2 = int(c), int(a), int(b)
            if p1 > 12:
                day, month = p1, p2
            elif p2 > 12:
                month, day = p1, p2
            else:
                # Ambíguo: assume dd/mm (padrão brasileiro)
                day, month = p1, p2
        else:
            return None

        dt = date(year, month, day)
        return dt.strftime("%d/%m/%Y")

    except (ValueError, TypeError):
        return None


IDADE_MIN = 17
IDADE_MAX = 96


def parse_idade(raw) -> float | None:
    """Retorna a idade como float, ou None se o valor for incoerente/ausente."""
    if pd.isna(raw):
        return None
    text = str(raw).strip()
    if text.lower() in INVALID_VALUES:
        return None
    try:
        return float(text)
    except (ValueError, TypeError):
        return None


def idade_valida(val) -> bool:
    """Verifica se a idade está dentro do intervalo aceitável."""
    try:
        return IDADE_MIN <= float(val) <= IDADE_MAX
    except (TypeError, ValueError):
        return False


def calcular_idade(data_str: str) -> float | None:
    """Calcula a idade a partir de uma string dd/mm/yyyy já normalizada."""
    try:
        dt = dateutil_parser.parse(data_str, dayfirst=True).date()
        hoje = date.today()
        idade = hoje.year - dt.year - ((hoje.month, hoje.day) < (dt.month, dt.day))
        return float(idade)
    except Exception:
        return None


MASCULINO = {"m", "masc", "masculino"}
FEMININO  = {"f", "fem", "feminino"}

def parse_genero(raw) -> str | None:
    """Normaliza gênero para 'M', 'F' ou None."""
    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    if text in MASCULINO:
        return "M"
    if text in FEMININO:
        return "F"
    return None  # qualquer outro valor vira NaN


CASADO   = {"c", "casado", "casado(a)", "married", "casado"}
SOLTEIRO = {"s", "solt", "solteiro", "solteiro(a)", "single"}

def parse_estado_civil(raw) -> str | None:
    """Normaliza estado civil para 'casado', 'solteiro' ou None."""
    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    if text in CASADO:
        return "casado"
    if text in SOLTEIRO:
        return "solteiro"
    return None


def tratar_base(df: pd.DataFrame) -> pd.DataFrame:
    """
    Recebe o DataFrame bruto e retorna ele tratado:
      - data_nascimento normalizada para dd/mm/yyyy
      - idade numérica, calculada a partir da data quando ausente ou fora do range
      - idades válidas: entre IDADE_MIN e IDADE_MAX anos
      - linhas onde AMBAS as colunas são incoerentes/inválidas são removidas
    """
    df = df.copy()

    # 1. Normaliza as colunas
    df["data_nascimento"] = df["data_nascimento"].apply(parse_date)
    df["idade"] = df["idade"].apply(parse_idade)
    if "genero" in df.columns:
        df["genero"] = df["genero"].apply(parse_genero)
    if "estado_civil" in df.columns:
        df["estado_civil"] = df["estado_civil"].apply(parse_estado_civil)

    # 2. Idade ausente ou fora do range → tenta calcular pela data de nascimento
    mask_corrigir = df["idade"].isna() | ~df["idade"].apply(
        lambda x: idade_valida(x) if pd.notna(x) else False
    )
    mask_tem_data = df["data_nascimento"].notna()

    mask_calcular = mask_corrigir & mask_tem_data
    df.loc[mask_calcular, "idade"] = df.loc[mask_calcular, "data_nascimento"].apply(calcular_idade)

    # 3. Remove linhas onde idade ainda está inválida/ausente E não há data de nascimento
    mask_deletar = ~df["idade"].apply(idade_valida) & df["data_nascimento"].isna()
    n_deletadas = mask_deletar.sum()
    df = df[~mask_deletar].reset_index(drop=True)

    print(f"Linhas removidas (sem dado aproveitável):     {n_deletadas}")
    print(f"Idades corrigidas via data_nascimento:        {mask_calcular.sum()}")
    return df


# ── Exemplo de uso ─────────────────────────────────────────────────────────────
if __name__ == "__main__":

    df = pd.read_csv("cadastro_clientes.csv")
    df = tratar_base(df)
    df.to_csv("cadastro_clientes_tratada.csv", index=False)

    # Dados de demonstração baseados nas prints
    dados_exemplo = {
        "idade":           [ 34.0,        36.0,  41.0,         18.0,         -5.0,         999.0,        40.0,          0.0,          "?",          None,  None,         37.0,         32.0,         45.0,         29.0        ],
        "data_nascimento": ["14/06/1972",  None, "11/06/1985", "05/06/2008", "10/06/1988", "15/06/1971", "06/11/1986", "08/06/1996", "11/06/1986", "14/06/1975", None, "10/06/1989", "09/06/1994", "12/06/1981", "08/06/1997"],
    }

    df_bruto = pd.DataFrame(dados_exemplo)

    print("=== Base bruta ===")
    print(df_bruto.to_string(index=False))
    print()

    df_tratado = tratar_base(df_bruto)

    print()
    print("=== Base tratada ===")
    print(df_tratado.to_string(index=False))

In [104]:
df["renda_anual"] = df["renda_anual"].astype(str).str.replace("R$", "", regex=False).str.strip()
print(df["renda_anual"].value_counts().head(50))

In [105]:
df.to_csv("cadastro_clientes_tratada.csv", index=False)

In [106]:
def converter_renda(val):
    if pd.isna(val):
        return None
    text = str(val).strip().replace("R$", "").replace(" ", "")
    # formato brasileiro: 100.000,00 → remove ponto, troca vírgula por ponto
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    try:
        return float(text)
    except ValueError:
        return None

df["renda_convertida"] = df["renda_anual"].apply(converter_renda)
print(df["renda_convertida"].describe())
print("\nMenores valores:")
print(df["renda_convertida"].nsmallest(20).tolist())
print("\nMaiores valores:")
print(df["renda_convertida"].nlargest(20).tolist())

In [107]:
print("Valores <= 1000:")
print(df[df["renda_convertida"] <= 1000]["renda_convertida"].value_counts().head(20).tolist())

print("\nValores entre 1000 e 10000:")
print(df[(df["renda_convertida"] > 1000) & (df["renda_convertida"] <= 10000)]["renda_convertida"].value_counts().head(20).tolist())

print("\nValores > 500000:")
print(df[df["renda_convertida"] > 500000]["renda_convertida"].value_counts().head(20).tolist())

In [108]:
import pandas as pd
import re
from dateutil import parser as dateutil_parser
from datetime import date

# ── Valores que indicam ausência de informação ─────────────────────────────────
INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}


def parse_date(raw) -> str | None:
    """
    Tenta interpretar qualquer string de data e retorna no formato dd/mm/yyyy.
    Retorna None se o valor for incoerente / ausente.
    """
    if pd.isna(raw):
        return None

    text = str(raw).strip()

    if text.lower() in INVALID_VALUES:
        return None

    # Formato textual: "11 Jun 1984"
    month_abbr = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
    if re.fullmatch(rf"\d{{1,2}} {month_abbr} \d{{4}}", text, re.IGNORECASE):
        try:
            dt = dateutil_parser.parse(text, dayfirst=True)
            return dt.strftime("%d/%m/%Y")
        except Exception:
            return None

    # Formatos numéricos
    parts = re.split(r"[-/.]", text)
    if len(parts) != 3:
        return None

    try:
        a, b, c = parts

        if len(a) == 4:
            year, p2, p3 = int(a), int(b), int(c)
            month, day = (p2, p3) if 1 <= p2 <= 12 else (p3, p2)

        elif len(c) == 4:
            year, p1, p2 = int(c), int(a), int(b)
            if p1 > 12:
                day, month = p1, p2
            elif p2 > 12:
                month, day = p1, p2
            else:
                # Ambíguo: assume dd/mm (padrão brasileiro)
                day, month = p1, p2
        else:
            return None

        dt = date(year, month, day)
        return dt.strftime("%d/%m/%Y")

    except (ValueError, TypeError):
        return None


IDADE_MIN = 17
IDADE_MAX = 96


def parse_idade(raw) -> float | None:
    """Retorna a idade como float, ou None se o valor for incoerente/ausente."""
    if pd.isna(raw):
        return None
    text = str(raw).strip()
    if text.lower() in INVALID_VALUES:
        return None
    try:
        return float(text)
    except (ValueError, TypeError):
        return None


def idade_valida(val) -> bool:
    """Verifica se a idade está dentro do intervalo aceitável."""
    try:
        return IDADE_MIN <= float(val) <= IDADE_MAX
    except (TypeError, ValueError):
        return False


def calcular_idade(data_str: str) -> float | None:
    """Calcula a idade a partir de uma string dd/mm/yyyy já normalizada."""
    try:
        dt = dateutil_parser.parse(data_str, dayfirst=True).date()
        hoje = date.today()
        idade = hoje.year - dt.year - ((hoje.month, hoje.day) < (dt.month, dt.day))
        return float(idade)
    except Exception:
        return None


MASCULINO = {"m", "masc", "masculino"}
FEMININO  = {"f", "fem", "feminino"}

def parse_genero(raw) -> str | None:
    """Normaliza gênero para 'M', 'F' ou None."""
    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    if text in MASCULINO:
        return "M"
    if text in FEMININO:
        return "F"
    return None  # qualquer outro valor vira NaN


CASADO   = {"c", "casado", "casado(a)", "married", "casado"}
SOLTEIRO = {"s", "solt", "solteiro", "solteiro(a)", "single"}

def parse_estado_civil(raw) -> str | None:
    """Normaliza estado civil para 'casado', 'solteiro' ou None."""
    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    if text in CASADO:
        return "casado"
    if text in SOLTEIRO:
        return "solteiro"
    return None


RENDA_MIN = 10_000
RENDA_MAX = 1_000_000

def parse_renda_anual(raw) -> float | None:
    """Converte renda anual para float, retorna None se inválida ou fora do range."""
    if pd.isna(raw):
        return None
    text = str(raw).strip().replace("R$", "").replace(" ", "")
    if text.lower() in INVALID_VALUES:
        return None
    # formato brasileiro (100.000,00): remove ponto de milhar, troca vírgula por ponto
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    # formato tipo "94.400" sem vírgula — ponto é separador de milhar se resultar em valor pequeno
    elif "." in text:
        try:
            val_direto = float(text)
            if val_direto < RENDA_MIN:  # provavelmente ponto de milhar, não decimal
                text = text.replace(".", "")
        except ValueError:
            pass
    try:
        val = float(text)
        return val if RENDA_MIN <= val <= RENDA_MAX else None
    except ValueError:
        return None


def tratar_base(df: pd.DataFrame) -> pd.DataFrame:
    """
    Recebe o DataFrame bruto e retorna ele tratado:
      - data_nascimento normalizada para dd/mm/yyyy
      - idade numérica, calculada a partir da data quando ausente ou fora do range
      - idades válidas: entre IDADE_MIN e IDADE_MAX anos
      - linhas onde AMBAS as colunas são incoerentes/inválidas são removidas
    """
    df = df.copy()

    # 1. Normaliza as colunas
    df["data_nascimento"] = df["data_nascimento"].apply(parse_date)
    df["idade"] = df["idade"].apply(parse_idade)
    if "genero" in df.columns:
        df["genero"] = df["genero"].apply(parse_genero)
    if "estado_civil" in df.columns:
        df["estado_civil"] = df["estado_civil"].apply(parse_estado_civil)
    if "renda_anual" in df.columns:
        df["renda_anual"] = df["renda_anual"].apply(parse_renda_anual)

    # 2. Idade ausente ou fora do range → tenta calcular pela data de nascimento
    mask_corrigir = df["idade"].isna() | ~df["idade"].apply(
        lambda x: idade_valida(x) if pd.notna(x) else False
    )
    mask_tem_data = df["data_nascimento"].notna()

    mask_calcular = mask_corrigir & mask_tem_data
    df.loc[mask_calcular, "idade"] = df.loc[mask_calcular, "data_nascimento"].apply(calcular_idade)

    # 3. Remove linhas onde idade ainda está inválida/ausente E não há data de nascimento
    mask_deletar = ~df["idade"].apply(idade_valida) & df["data_nascimento"].isna()
    n_deletadas = mask_deletar.sum()
    df = df[~mask_deletar].reset_index(drop=True)

    print(f"Linhas removidas (sem dado aproveitável):     {n_deletadas}")
    print(f"Idades corrigidas via data_nascimento:        {mask_calcular.sum()}")
    return df


# ── Exemplo de uso ─────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # Substitua pelo caminho real do seu arquivo:
    df = pd.read_csv("cadastro_clientes_tratada.csv")
    df = tratar_base(df)
    df.to_csv("cadastro_clientes_tratada.csv", index=False)

    # Dados de demonstração baseados nas prints
    dados_exemplo = {
        "idade":           [ 34.0,        36.0,  41.0,         18.0,         -5.0,         999.0,        40.0,          0.0,          "?",          None,  None,         37.0,         32.0,         45.0,         29.0        ],
        "data_nascimento": ["14/06/1972",  None, "11/06/1985", "05/06/2008", "10/06/1988", "15/06/1971", "06/11/1986", "08/06/1996", "11/06/1986", "14/06/1975", None, "10/06/1989", "09/06/1994", "12/06/1981", "08/06/1997"],
    }

    print(f"max: {df['renda_anual'].max()} | min: {df['renda_anual'].min()}") 

In [109]:
df['possui_imovel'].value_counts()

In [110]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "N/D"}

def parse_possui_imovel(raw) -> str | None:
    if pd.isna(raw):
        return None
    text = str(raw).strip()
    if text.lower() in INVALID_VALUES:
        return None

    df.to_csv("cadastro_clientes_tratada.csv", index=False)

In [111]:
df.to_csv("cadastro_clientes_tratada")

In [112]:
df['possui_imovel'].value_counts()

In [113]:
import numpy as np

df['possui_imovel'] = (
    df['possui_imovel']
    .astype(str)          # garante que tudo é string
    .str.strip()          # remove espaços
    .replace(["#N/D", "-", "?", ""], np.nan)
)

In [114]:
df['possui_imovel'].value_counts(dropna=False)

In [115]:
import numpy as np

df['possui_imovel'] = (
    df['possui_imovel']
    .replace(["#N/D", "-", "?", "nan"], np.nan)
)

In [116]:
df['possui_imovel'].value_counts(dropna=False)

In [117]:
df.to_csv("cadastro_clientes_tratada.csv")

In [58]:
import pandas as pd

df = pd.read_csv('contratos_apolices.csv', sep=';')
df['num_apolices_ativas'].value_counts()

In [59]:
import numpy as np

df['num_apolices_ativas'] = (
    df['num_apolices_ativas']
    .astype(str)          # garante consistência
    .str.strip()          # remove espaços invisíveis
    .replace(["#N/D", "-", "?", ""], np.nan)  # lixo → NaN
    .astype(float)        # converte pra número
)

In [60]:
df['num_apolices_ativas'].value_counts()

In [61]:
df['tipo_cobertura'].value_counts()

In [62]:
import numpy as np

df['tipo_cobertura'] = (
    df['tipo_cobertura']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        # PADRAO
        "padrao": "padrao",
        "padrão": "padrao",
        "std": "padrao",

        # BASICA
        "basica": "basica",
        "básica": "basica",
        "basic": "basica",

        # PREMIUM
        "premium": "premium",
        "plus": "premium",
        "prem.": "premium",

        # lixo → NaN
        "?": np.nan,
        "#n/d": np.nan,
        "-": np.nan,
        "": np.nan
    })
)

In [63]:
df['tipo_cobertura'].value_counts()

In [64]:
df.to_csv('contratos_apolices_tratada.csv')

In [118]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}
PREMIO_MIN = 100
PREMIO_MAX = 20_000


def parse_premio_anual(raw) -> float | None:
    """Converte valor_premio_anual para float, retorna None se inválido ou fora do range."""
    if pd.isna(raw):
        return None
    text = str(raw).strip().replace("R$", "").replace(" ", "")
    if text.lower() in INVALID_VALUES:
        return None
    # formato brasileiro (1.500,00): remove ponto de milhar, troca vírgula por ponto
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    # formato tipo "1.500" sem vírgula — ponto é separador de milhar se resultar em valor pequeno
    elif "." in text:
        try:
            val_direto = float(text)
            if val_direto < PREMIO_MIN:
                text = text.replace(".", "")
        except ValueError:
            pass
    try:
        val = float(text)
        return val if PREMIO_MIN <= val <= PREMIO_MAX else None
    except ValueError:
        return None


if __name__ == "__main__":
    df = pd.read_csv("contratos_apolices.csv", sep=";")
    df["valor_premio_anual"] = df["valor_premio_anual"].apply(parse_premio_anual)
    df.to_csv("contratos_apolices_tratada.csv", index=False)
    print("Concluído!")
    print(df["valor_premio_anual"].describe())

In [119]:
import numpy as np
import pandas as pd

df = pd.read_csv('contratos_apolices_tratada.csv', sep=",")

df['tipo_cobertura'] = (
    df['tipo_cobertura']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        # PADRAO
        "padrao": "padrao",
        "padrão": "padrao",
        "std": "padrao",

        # BASICA
        "basica": "basica",
        "básica": "basica",
        "basic": "basica",

        # PREMIUM
        "premium": "premium",
        "plus": "premium",
        "prem.": "premium",

        # lixo → NaN
        "?": np.nan,
        "#n/d": np.nan,
        "-": np.nan,
        "": np.nan
    })
)

df.to_csv('contratos_apolices_tratada.csv')

In [120]:
df['tipo_cobertura'].value_counts()

In [121]:
import numpy as np
import pandas as pd

df = pd.read_csv('contratos_apolices_tratada.csv', sep=",")

df['num_apolices_ativas'] = (
    df['num_apolices_ativas']
    .astype(str)          # garante consistência
    .str.strip()          # remove espaços invisíveis
    .replace(["#N/D", "-", "?", ""], np.nan)  # lixo → NaN
    .astype(float)        # converte pra número
)

df.to_csv('contratos_apolices_tratada.csv')

In [122]:
import numpy as np
import pandas as pd

df = pd.read_csv('contratos_apolices_tratada.csv', sep=",")

df['tempo_cliente_dias'] = (
    df['tempo_cliente_dias']
    .astype(str)          # garante consistência
    .str.strip()          # remove espaços invisíveis
    .replace(["#N/D", "-", "?", ""], np.nan)  # lixo → NaN
    .astype(float)        # converte pra número
)

df.to_csv('contratos_apolices_tratada.csv')

In [123]:
import pandas as pd
import re
from dateutil import parser as dateutil_parser
from datetime import date

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}
DIAS_MIN = 29
DIAS_MAX = 8001


# ── Reutilizado do script de datas ────────────────────────────────────────────
def parse_date(raw) -> str | None:
    """Normaliza data para dd/mm/yyyy, retorna None se incoerente."""
    if pd.isna(raw):
        return None
    text = str(raw).strip()
    if text.lower() in INVALID_VALUES:
        return None

    month_abbr = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
    if re.fullmatch(rf"\d{{1,2}} {month_abbr} \d{{4}}", text, re.IGNORECASE):
        try:
            return dateutil_parser.parse(text, dayfirst=True).strftime("%d/%m/%Y")
        except Exception:
            return None

    parts = re.split(r"[-/.]", text)
    if len(parts) != 3:
        return None

    try:
        a, b, c = parts
        if len(a) == 4:
            year, p2, p3 = int(a), int(b), int(c)
            month, day = (p2, p3) if 1 <= p2 <= 12 else (p3, p2)
        elif len(c) == 4:
            year, p1, p2 = int(c), int(a), int(b)
            if p1 > 12:
                day, month = p1, p2
            elif p2 > 12:
                month, day = p1, p2
            else:
                day, month = p1, p2  # ambíguo: assume dd/mm
        else:
            return None
        return date(year, month, day).strftime("%d/%m/%Y")
    except (ValueError, TypeError):
        return None


def calcular_dias(data_str: str) -> float | None:
    """Calcula quantos dias se passaram desde a data até hoje."""
    if not data_str:
        return None
    try:
        dt = dateutil_parser.parse(data_str, dayfirst=True).date()
        delta = (date.today() - dt).days
        return float(delta) if DIAS_MIN <= delta <= DIAS_MAX else None
    except Exception:
        return None


def dias_valido(val) -> bool:
    try:
        return DIAS_MIN <= float(val) <= DIAS_MAX
    except (TypeError, ValueError):
        return False


def parse_dias(raw) -> float | None:
    """Retorna tempo_cliente_dias como float, ou None se incoerente/fora do range."""
    if pd.isna(raw):
        return None
    text = str(raw).strip()
    if text.lower() in INVALID_VALUES:
        return None
    try:
        val = float(text)
        return val if DIAS_MIN <= val <= DIAS_MAX else None
    except (ValueError, TypeError):
        return None


def tratar_tempo_cliente(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Normaliza a data
    df["data_primeira_apolice"] = df["data_primeira_apolice"].apply(parse_date)

    # 2. Normaliza dias (já valida o range)
    df["tempo_cliente_dias"] = df["tempo_cliente_dias"].apply(parse_dias)

    # 3. Onde dias está ausente ou fora do range → calcula pela data
    mask_corrigir = df["tempo_cliente_dias"].isna()
    mask_tem_data = df["data_primeira_apolice"].notna()
    mask_calcular = mask_corrigir & mask_tem_data

    df.loc[mask_calcular, "tempo_cliente_dias"] = (
        df.loc[mask_calcular, "data_primeira_apolice"].apply(calcular_dias)
    )

    # 4. Remove linhas onde ambas continuam sem info válida
    mask_deletar = df["tempo_cliente_dias"].isna() & df["data_primeira_apolice"].isna()
    n_deletadas = mask_deletar.sum()
    df = df[~mask_deletar].reset_index(drop=True)

    print(f"Linhas removidas (sem dado aproveitável):  {n_deletadas}")
    print(f"Dias calculados via data_primeira_apolice: {mask_calcular.sum()}")
    return df


if __name__ == "__main__":
    df = pd.read_csv("contratos_apolices_tratada.csv")
    df = tratar_tempo_cliente(df)
    df.to_csv("contratos_apolices_tratada.csv", index=False)
    print("\nConcluído!")
    print(df["tempo_cliente_dias"].describe())

In [124]:
df['num_produtos_contratados'].value_counts()

In [125]:
import pandas as pd

df = pd.read_csv('contratos_apolices_tratada.csv')

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "N/D"}

def parse_num_produtos_contratados(raw) -> str | None:
    if pd.isna(raw):
        return None
    text = str(raw).strip()
    if text.lower() in INVALID_VALUES:
        return None

df.to_csv('contratos_apolices_tratada.csv', index=False)

In [126]:
import numpy as np
import pandas as pd

df = pd.read_csv('contratos_apolices_tratada.csv')

df['num_produtos_contratados'] = (
    df['num_produtos_contratados']
    .astype(str)          # garante consistência
    .str.strip()          # remove espaços invisíveis
    .replace(["#N/D", "-", "?", ""], np.nan)  # lixo → NaN
    .astype(float)        # converte pra número
)

df.to_csv('contratos_apolices_tratada.csv')

In [129]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

COBERTURA_MIN = 30_000
COBERTURA_MAX = 1_200_000
FRANQUIA_MIN  = 100
FRANQUIA_MAX  = 5_000


def _parse_valor(raw, vmin: float, vmax: float) -> float | None:
    """Converte string monetária para float e valida o range."""
    if pd.isna(raw):
        return None
    text = str(raw).strip().replace("R$", "").replace(" ", "")
    if text.lower() in INVALID_VALUES:
        return None
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    elif "." in text:
        try:
            if float(text) < vmin:
                text = text.replace(".", "")
        except ValueError:
            pass
    try:
        val = float(text)
        return val if vmin <= val <= vmax else None
    except ValueError:
        return None


def parse_cobertura_total(raw) -> float | None:
    return _parse_valor(raw, COBERTURA_MIN, COBERTURA_MAX)


def parse_franquia_media(raw) -> float | None:
    return _parse_valor(raw, FRANQUIA_MIN, FRANQUIA_MAX)


if __name__ == "__main__":
    df = pd.read_csv("contratos_apolices_tratada.csv")

    df["valor_cobertura_total"] = df["valor_cobertura_total"].apply(parse_cobertura_total)
    df["franquia_media"]        = df["franquia_media"].apply(parse_franquia_media)

    df.to_csv("contratos_apolices_tratada.csv", index=False)
    print("Concluído!")
    print("\nvalor_cobertura_total:")
    print(df["valor_cobertura_total"].describe())
    print("\nfranquia_media:")
    print(df["franquia_media"].describe())

In [ ]:
import pandas as pd

df = pd.read_csv('contratos_apolices_tratada.csv')

df.drop(['Unnamed: 0', 'Unnamed: 0.1', 'Unnamed: 0.2', 'Unnamed: 0.3', 'Unnamed: 0.4'], axis=1)

df = pd.to_csv('contratos_apolices_tratada.csv')

In [3]:
import pandas as pd
df = pd.read_csv('contratos_apolices_tratada.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

In [4]:
df.to_csv('contratos_apolices_tratada.csv', index=False)

In [5]:
df = pd.read_csv('contratos_apolices_tratada.csv')
print(df.columns)

Index(['cod_individuo', 'num_apolices_ativas', 'tipo_cobertura',
       'valor_premio_anual', 'tempo_cliente_dias', 'data_primeira_apolice',
       'num_produtos_contratados', 'valor_cobertura_total', 'franquia_media',
       'canal_aquisicao', 'pagamento_em_dia', 'desconto_aplicado_pct',
       'metodo_pagamento'],
      dtype='object')


In [ ]:
import pandas as pd
df = pd.read_csv('Base_Unificada_Tratada.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

df.to_csv('Base_Unificada_Tratada.csv', index=False)

print(df.columns)

In [7]:
df.to_csv('cadastro_clientes_tratada.csv', index=False)

In [8]:
df = pd.read_csv('cadastro_clientes_tratada.csv')
print(df.columns)

Index(['id_cliente', 'idade', 'data_nascimento', 'genero', 'estado_civil',
       'tem_filhos', 'qtd_dependentes', 'escolaridade', 'renda_anual',
       'possui_imovel', 'valor_imovel', 'tempo_residencia_anos'],
      dtype='object')


In [10]:
df = pd.read_csv('contratos_apolices_tratada.csv')
df["canal_aquisicao"].value_counts()

canal_aquisicao
Agente         25745
Digital        21423
Telefone       14201
Indicacao       9884
 Agente         1630
  Agente        1604
Agente          1567
Agente          1548
  Digital       1356
 Digital        1331
Digital         1330
Digital         1293
Telefone         903
 Telefone        895
  Telefone       891
Telefone         849
Indicacao        649
Indicacao        633
 Indicacao       612
  Indicacao      602
                 490
                 490
?                488
-                475
#N/D             447
Name: count, dtype: int64

In [11]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

CANAIS = {"agente", "digital", "telefone", "indicacao", "indicação"}


def parse_canal_aquisicao(raw) -> str | None:
    """Normaliza canal_aquisicao para valor padronizado ou None."""
    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    if text in CANAIS:
        # capitaliza a primeira letra (ex: "agente" → "Agente")
        return text.replace("indicação", "Indicacao").capitalize()
    return None


if __name__ == "__main__":
    df = pd.read_csv("contratos_apolices_tratada.csv")
    df["canal_aquisicao"] = df["canal_aquisicao"].apply(parse_canal_aquisicao)
    print(df["canal_aquisicao"].value_counts(dropna=False))
    df.to_csv("contratos_apolices_tratada.csv", index=False)
    print("\nConcluído!")

canal_aquisicao
Agente       32094
Digital      26733
Telefone     17739
Indicacao    12380
None          4792
Name: count, dtype: int64



Concluído!


In [12]:
df = pd.read_csv('contratos_apolices_tratada.csv')
df["pagamento_em_dia"].value_counts()

pagamento_em_dia
1           16833
OK          16716
S           16665
Sim         16596
Em dia      16551
N            1479
Atrasado     1470
0            1425
Nao          1373
#N/D          492
?             471
              452
              451
-             430
Name: count, dtype: int64

In [25]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

Sim = {"1", "S", "OK", "Sim", "Em dia"}
Nao = {"N", "Atrasado", "0", "Nao"}
 
def parse_pagamento_em_dia(raw) -> str | None:

    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    if text in Sim:
        return "Sim"
    if text in Nao:
        return "Nao"
    return None


if __name__ == "__main__":
    df = pd.read_csv("contratos_apolices_tratada.csv")
    df["pagamento_em_dia"] = df["pagamento_em_dia"].apply(parse_pagamento_em_dia)
    print(df["pagamento_em_dia"].value_counts(dropna=False))
    df.to_csv("contratos_apolices_tratada.csv", index=False)
    print("\nConcluído!")

pagamento_em_dia
None    75480
Sim     16833
Nao      1425
Name: count, dtype: int64



Concluído!


In [33]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

def parse_desconto_aplicado_pct(raw) -> str | None:

    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    return raw

if __name__ == "__main__":
    df = pd.read_csv("contratos_apolices_tratada.csv")
    df["desconto_aplicado_pct"] = df["desconto_aplicado_pct"].apply(parse_desconto_aplicado_pct)
    print(df["desconto_aplicado_pct"].value_counts(dropna=False))
    df.to_csv("contratos_apolices_tratada.csv", index=False)
    print("\nConcluído!")

desconto_aplicado_pct
None     4645
0.049     709
0.048     691
0.0       680
0.046     679
         ... 
0.243       7
0.246       7
0.248       5
0.247       4
0.249       3
Name: count, Length: 252, dtype: int64



Concluído!


In [36]:
import pandas as pd

# 1. Carregue os seus arquivos (mude para .csv ou .xlsx dependendo do seu caso)
df_principal = pd.read_csv('contratos_apolices_tratada.csv')
df_fonte = pd.read_csv('contratos_apolices.csv', sep=";")

# 2. Crie o mapeamento automático na memória
# Substitua 'ID_PRODUTO' e 'PRECO_NOVO' pelos nomes reais das suas colunas
mapeamento = df_fonte.set_index('cod_individuo')['pagamento_em_dia']

# 3. Substitua a coluna na tabela principal usando a referência
# Substitua 'PRECO_ANTIGO' pelo nome da coluna que você quer atualizar
df_principal['pagamento_em_dia'] = df_principal['cod_individuo'].map(mapeamento)

# 4. (Opcional) Salve o resultado em um novo arquivo
df_principal.to_csv('contratos_apolices_tratada.csv', index=False)

print("Substituição concluída com sucesso!")

df_principal['pagamento_em_dia'].value_counts()


Substituição concluída com sucesso!


pagamento_em_dia
1           16833
OK          16716
S           16665
Sim         16596
Em dia      16551
N            1479
Atrasado     1470
0            1425
Nao          1373
#N/D          492
?             471
              452
              451
-             430
Name: count, dtype: int64

In [30]:
import pandas as pd

df = pd.read_csv('contratos_apolices_tratada.csv')

df['metodo_pagamento'].value_counts()

metodo_pagamento
debito     33893
boleto     22149
credito    19750
pix        13290
Name: count, dtype: int64

In [13]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

debito = {"debito", "deb auto", "debito_auto", "debito automatico"}
boleto = {"boleto bancario", "boleto", "bol"}
credito = {"cartao", "cc", "cartao_credito", "cartão"}
pix = {"pix"}

def parse_metodo_pagamento(raw):
    if pd.isna(raw):
        return None
    
    text = str(raw).strip().lower()
    
    if text in INVALID_VALUES:
        return None
    if text in debito:
        return "debito"
    if text in boleto:
        return "boleto"
    if text in credito:
        return "credito"
    if text in pix:
        return "pix"
    
    return None

df = pd.read_csv("contratos_apolices_tratada.csv")

df["metodo_pagamento"] = df["metodo_pagamento"].apply(parse_metodo_pagamento)

print(df["metodo_pagamento"].value_counts(dropna=False))

df.to_csv("contratos_apolices_tratada.csv", index=False)

metodo_pagamento
debito     33893
boleto     22149
credito    19750
pix        13290
None        4656
Name: count, dtype: int64


In [15]:
import pandas as pd

df = pd.read_csv('cadastro_clientes_tratada.csv')

df['tem_filhos'].value_counts()

tem_filhos
1        8384
sim      8279
Sim      8266
true     8208
SIM      8149
S        8142
false    7447
Nao      7442
0        7442
NAO      7411
N        7312
nao      7295
          502
#N/D      498
?         498
          469
-         466
Name: count, dtype: int64

In [12]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

SIM = {"1", "sim", "s", "true"}
NAO = {"0", "nao", "não", "n", "false"}


def parse_tem_filhos(raw) -> str | None:
    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    if text in SIM:
        return "sim"
    if text in NAO:
        return "nao"
    return None


if __name__ == "__main__":
    df = pd.read_csv("cadastro_clientes_tratada.csv", keep_default_na=False, na_values=[""])
    df["tem_filhos"] = df["tem_filhos"].apply(parse_tem_filhos)
    print(df["tem_filhos"].value_counts(dropna=False))
    df.to_csv("cadastro_clientes_tratada.csv", index=False)
    print("\nConcluído!")

tem_filhos
sim     49428
nao     44349
None     4930
Name: count, dtype: int64



Concluído!


In [3]:
df_raw = pd.read_csv("cadastro_clientes_tratada.csv")
nulos = df_raw[df_raw["tem_filhos"].apply(parse_tem_filhos).isna()]
print(nulos["tem_filhos"].value_counts(dropna=False).head(30))

tem_filhos
NaN    20384
Name: count, dtype: int64


In [4]:
df_raw = pd.read_csv("cadastro_clientes.csv")
print(df_raw["tem_filhos"].isna().sum())

2504


In [6]:
df_raw = pd.read_csv("cadastro_clientes.csv")
print(df_raw["tem_filhos"].value_counts(dropna=False))

tem_filhos
1        8402
sim      8307
Sim      8289
true     8228
S        8172
SIM      8172
false    7477
Nao      7467
0        7460
NAO      7429
N        7339
nao      7313
NaN      2504
          503
#N/D      501
?         501
          470
-         466
Name: count, dtype: int64


In [10]:
df = pd.read_csv("cadastro_clientes_tratada.csv")
print(df["tem_filhos"].isna().sum())

2497


In [11]:
df = pd.read_csv("cadastro_clientes_tratada.csv")
print(df["tem_filhos"].value_counts(dropna=False).head(30))

tem_filhos
1        8384
sim      8279
Sim      8266
true     8208
SIM      8149
S        8142
false    7447
Nao      7442
0        7442
NAO      7411
N        7312
nao      7295
NaN      2497
          502
#N/D      498
?         498
          469
-         466
Name: count, dtype: int64


In [13]:
df = pd.read_csv("cadastro_clientes_tratada.csv", keep_default_na=False, na_values=[""])
nulos = df[df["tem_filhos"].apply(parse_tem_filhos).isna()]
print(nulos["tem_filhos"].value_counts(dropna=False).head(30))

tem_filhos
NaN    4930
Name: count, dtype: int64


In [14]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

#COBERTURA_MIN = 30_000
#COBERTURA_MAX = 1_200_000
FRANQUIA_MIN  = 50_000
FRANQUIA_MAX  = 1_500_000


def _parse_valor(raw, vmin: float, vmax: float) -> float | None:
    """Converte string monetária para float e valida o range."""
    if pd.isna(raw):
        return None
    text = str(raw).strip().replace("R$", "").replace(" ", "")
    if text.lower() in INVALID_VALUES:
        return None
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    elif "." in text:
        try:
            if float(text) < vmin:
                text = text.replace(".", "")
        except ValueError:
            pass
    try:
        val = float(text)
        return val if vmin <= val <= vmax else None
    except ValueError:
        return None


#def parse_cobertura_total(raw) -> float | None:
 #   return _parse_valor(raw, COBERTURA_MIN, COBERTURA_MAX)


#def parse_franquia_media(raw) -> float | None:
  #  return _parse_valor(raw, FRANQUIA_MIN, FRANQUIA_MAX)

def parse_franquia_media(raw) -> float | None:
    return _parse_valor(raw, FRANQUIA_MIN, FRANQUIA_MAX)


if __name__ == "__main__":
    df = pd.read_csv("cadastro_clientes_tratada.csv")

    #df["valor_cobertura_total"] = df["valor_cobertura_total"].apply(parse_cobertura_total)
    #df["franquia_media"]        = df["franquia_media"].apply(parse_franquia_media)
    df["valor_imovel"]        = df["valor_imovel"].apply(parse_franquia_media)

    df.to_csv("cadastro_clientes_tratada.csv", index=False)
    print("Concluído!")
    print("\nvalor_imovel:")
    print(df["valor_imovel"].describe())
    print("\nvalor_imovel:")
    print(df["valor_imovel"].describe())

Concluído!

valor_imovel:
count    9.282500e+04
mean     2.995235e+05
std      1.227869e+05
min      5.000000e+04
25%      2.240000e+05
50%      2.920000e+05
75%      3.600000e+05
max      1.500000e+06
Name: valor_imovel, dtype: float64

valor_imovel:
count    9.282500e+04
mean     2.995235e+05
std      1.227869e+05
min      5.000000e+04
25%      2.240000e+05
50%      2.920000e+05
75%      3.600000e+05
max      1.500000e+06
Name: valor_imovel, dtype: float64


In [37]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

Sim = {"1", "s", "ok", "sim", "em dia"}
Nao = {"n", "atrasado", "0", "nao"}
 
def parse_pagamento_em_dia(raw) -> str | None:

    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    if text in Sim:
        return "Sim"
    if text in Nao:
        return "Nao"
    return None


if __name__ == "__main__":
    df = pd.read_csv("contratos_apolices_tratada.csv")
    df["pagamento_em_dia"] = df["pagamento_em_dia"].apply(parse_pagamento_em_dia)
    print(df["pagamento_em_dia"].value_counts(dropna=False))
    df.to_csv("contratos_apolices_tratada.csv", index=False)
    print("\nConcluído!")

pagamento_em_dia
Sim     83361
Nao      5747
None     4630
Name: count, dtype: int64



Concluído!


In [25]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

df = pd.read_csv('Engajamento_Marketing_Tratado_Final.csv')

def parse_indicou_clientes(x):
    if pd.isna(x):
        return None
    
    x = str(x).strip().lower()
    
    if x in INVALID_VALUES:
        return None
    
    if x == "0":
        return 0
    elif x == "1":
        return 1
    else:
        return None

# aplicar
df["indicou_clientes"] = df["indicou_clientes"].apply(parse_indicou_clientes)

df.to_csv('Engajamento_Marketing_Tratado_Final.csv', index=False)

# checar
print(df["indicou_clientes"].value_counts(dropna=False))

indicou_clientes
0.0    76971
1.0    13269
NaN     4760
Name: count, dtype: int64


COLOCANDO VALORES BINARIOS (SIM, NAO) EM BINARIO (0, 1)

In [35]:
import pandas as pd

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

Sim = {"Sim"}
Nao = {"Nao"}
 
def parse_pagamento_em_dia(raw) -> str | None:

    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    if text in Sim:
        return "1"
    if text in Nao:
        return "0"
    return None


if __name__ == "__main__":
    df = pd.read_csv("contratos_apolices_tratada.csv")
    df["pagamento_em_dia"] = df["pagamento_em_dia"].apply(parse_pagamento_em_dia)
    print(df["pagamento_em_dia"].value_counts(dropna=False))
    df.to_csv("contratos_apolices_tratada.csv", index=False)
    print("\nConcluído!")


Concluído!


In [5]:
import pandas as pd

df = pd.read_csv('contratos_apolices_tratada.csv')

def parse_sim_nao(x):
    if pd.isna(x):
        return x  # mantém NaN
    
    x = str(x).strip().lower()
    
    if x == "sim":
        return 1
    elif x == "nao":
        return 0
    else:
        return None  # caso apareça lixo inesperado

df["pagamento_em_dia"] = df["pagamento_em_dia"].apply(parse_sim_nao)

df.to_csv('contratos_apolices_tratada.csv', index=False)

print(df["pagamento_em_dia"].value_counts(dropna=False))

In [4]:
import pandas as pd

df = pd.read_csv('cadastro_clientes_tratada.csv')

def parse_sim_nao(x):
    if pd.isna(x):
        return x  # mantém NaN
    
    x = str(x).strip().lower()
    
    if x == "m":
        return 1
    elif x == "f":
        return 0
    else:
        return None  # caso apareça lixo inesperado

df["genero"] = df["genero"].apply(parse_sim_nao)

df.to_csv('cadastro_clientes_tratada.csv', index=False)

print(df["genero"].value_counts(dropna=False))

In [3]:
import pandas as pd

df = pd.read_csv('cadastro_clientes_tratada.csv')

def parse_sim_nao(x):
    if pd.isna(x):
        return x  # mantém NaN
    
    x = str(x).strip().lower()
    
    if x == "solteiro":
        return 1
    elif x == "casado":
        return 0
    else:
        return None  # caso apareça lixo inesperado

df["estado_civil"] = df["estado_civil"].apply(parse_sim_nao)

df.to_csv('cadastro_clientes_tratada.csv', index=False)

print(df["estado_civil"].value_counts(dropna=False))

In [2]:
import pandas as pd

df = pd.read_csv('cadastro_clientes_tratada.csv')

def parse_sim_nao(x):
    if pd.isna(x):
        return x  # mantém NaN
    
    x = str(x).strip().lower()
    
    if x == "sim":
        return 1
    elif x == "nao":
        return 0
    else:
        return None  # caso apareça lixo inesperado

df["tem_filhos"] = df["tem_filhos"].apply(parse_sim_nao)

df.to_csv('cadastro_clientes_tratada.csv', index=False)

print(df["tem_filhos"].value_counts(dropna=False))

In [9]:
df.to_csv('Base_Unificada_Tratada.csv')

Gerando o Gráfico Divergente para Apresentação Executiva...



NameError: name 'df_final_eda' is not defined

In [26]:
import pandas as pd
df = pd.read_csv('Engajamento_Marketing_Tratado_Final.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

df.to_csv('Engajamento_Marketing_Tratado_Final.csv', index=False)

df = pd.read_csv('Engajamento_Marketing_Tratado_Final.csv')
print(df.columns)

Index(['cod_individuo', 'score_engajamento_digital', 'indicou_clientes',
       'renovacoes_consecutivas', 'indice_relacionamento', 'tipo_veiculo',
       'ano_veiculo', 'km_anual_estimado', 'segmento_marketing',
       'regiao_vendas', 'ultimo_login_portal_dias', 'score_propensao_churn',
       'cluster_sugerido_crm'],
      dtype='object')


In [1]:
import pandas as pd

df = pd.read_csv('cadastro_clientes_tratada.csv')

df['escolaridade'].value_counts()

escolaridade
Medio              19439
Superior           18110
Fundamental         5900
Pos                 5865
  Medio             3970
Medio               3906
 Medio              3779
Medio               3702
Superior            3612
  Superior          3598
Superior            3375
 Superior           3220
Fundamental         1223
Pos                 1179
  Pos               1174
  Fundamental       1161
 Pos                1134
Fundamental         1124
 Fundamental        1113
Pos                 1111
                     519
-                    483
?                    480
                     474
#N/D                 472
 Medio               399
  Medio              370
 Medio               366
  Medio              359
 Superior            359
   Medio             346
   Superior          322
  Superior           320
Superior             318
Medio                307
 Superior            292
  Superior           286
    Medio            190
    Superior         175
Superior    

In [3]:
import pandas as pd 

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

ESCOLARIDADE = {"medio", "fundamental", "superior", "pos"}


def parse_escolaridade(raw) -> str | None:
    """Normaliza canal_aquisicao para valor padronizado ou None."""
    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    if text in ESCOLARIDADE:
        # capitaliza a primeira letra (ex: "agente" → "Agente")
        return text.replace("medio", "Medio").capitalize()
    return None


if __name__ == "__main__":
    df = pd.read_csv("cadastro_clientes_tratada.csv")
    df["escolaridade"] = df["escolaridade"].apply(parse_escolaridade)
    print(df["escolaridade"].value_counts(dropna=False))
    df.to_csv("cadastro_clientes_tratada.csv", index=False)
    print("\nConcluído!")

escolaridade
Medio          37292
Superior       34148
Fundamental    11295
Pos            11171
None            4801
Name: count, dtype: int64



Concluído!


In [8]:
import pandas as pd 

INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d"}

coluna = "tempo_residencia_anos"

#ESCOLARIDADE = {"medio", "fundamental", "superior", "pos"}

def parse_tempo_residencia_anos(raw) -> str | None:
    if pd.isna(raw):
        return None
    text = str(raw).strip().lower()
    if text in INVALID_VALUES:
        return None
    #if text in ESCOLARIDADE:
        # capitaliza a primeira letra (ex: "agente" → "Agente")
        #return text.replace("medio", "Medio").capitalize()
    return raw


if __name__ == "__main__":
    df = pd.read_csv("cadastro_clientes_tratada.csv")
    df[coluna] = df[coluna].apply(parse_tempo_residencia_anos)
    print(df[coluna].value_counts(dropna=False))
    df.to_csv("cadastro_clientes_tratada.csv", index=False)
    print("\nConcluído!")

tempo_residencia_anos
1       11577
2       10013
3        8753
4        7579
5        6664
0        6480
6        5777
None     4964
7        4869
8        4213
9        3699
10       3260
11       2754
12       2390
13       2106
14       1808
15       1534
16       1364
17       1151
18       1050
19        889
20        755
21        694
22        578
23        520
24        445
25        357
26        338
40        317
27        312
28        254
29        208
30        188
31        163
32        135
33        110
34         99
35         99
36         75
37         69
38         57
39         40
Name: count, dtype: int64



Concluído!


In [20]:
df = pd.read_csv('Atendimento_Sinistros_Tratado_Final.csv')

ids_repetidos = df[df["cod_individuo"].duplicated(keep=False)]

print(ids_repetidos.sort_values("cod_individuo"))

       cod_individuo  num_reclamacoes_12m  num_sinistros_historico  \
26610   221300001143                  0.0                      0.0   
54845   221300001143                  0.0                      0.0   
89776   221300002960                  1.0                      1.0   
85305   221300002960                  1.0                      1.0   
82348   221300004045                  1.0                      0.0   
...              ...                  ...                      ...   
91548   221303306551                  0.0                      2.5   
39791   221303306866                  1.0                      2.0   
2648    221303306866                  1.0                      2.0   
73253   221303307284                  1.0                      0.0   
71222   221303307284                  1.0                      0.0   

       dias_ultimo_contato  tempo_medio_resposta_dias  \
26610                 35.0                        0.2   
54845                 35.0                   

In [27]:
import pandas as pd
import numpy as np

df = pd.read_csv('contratos_apolices_tratada.csv')

print("Iniciando limpeza e transformação de 'metodo_pagamento'...\n")

# A nossa clássica lista de sujeiras
lixos = ['#N/D', 'N/A', 'NA', '-', '?', '', 'Desconhecido', 'null', 'NULL', 'nan']

# Trava de segurança para evitar o erro de coluna não encontrada
if 'metodo_pagamento' in df.columns:
    
    # 1. FAXINA SEMÂNTICA
    df['metodo_pagamento'] = df['metodo_pagamento'].astype(str).str.strip()
    df['metodo_pagamento'] = df['metodo_pagamento'].replace(lixos, 'NaN')
    
    print("✅ Sujeiras removidas! Valores únicos que sobraram no segmento:")
    print(df['metodo_pagamento'].unique(), "\n")
    
    # 2. FATIAMENTO BINÁRIO (ONE-HOT ENCODING)
    df = pd.get_dummies(df, columns=['metodo_pagamento'], prefix='metodo_pagamento', dtype=int)
    print("✅ Coluna transformada em binários com sucesso!")

else:
    print("⚠️ A coluna 'metodo_pagamento' não foi encontrada. Você precisa rodar o Motor de Limpeza de novo para ela reaparecer!")

# 3. EXIBIÇÃO E SALVAMENTO
#colunas_segmento = [col for col in df_mkt_final.columns if 'escolaridade' in col]

#if colunas_segmento:
    #print("\nNovas colunas geradas:")
    #print(colunas_segmento, "\n")
    
    # Atualiza o arquivo final
df.to_csv('contratos_apolices_tratada.csv', index=False)
    #print("💾 Arquivo 'cadastro_clientes_tratada.csv' atualizado e salvo no disco!\n")
    
    # Mostra o resultado final
display(df['metodo_pagamento'].head())

Iniciando limpeza e transformação de 'metodo_pagamento'...

✅ Sujeiras removidas! Valores únicos que sobraram no segmento:
['boleto' 'credito' 'debito' 'pix' 'NaN'] 

✅ Coluna transformada em binários com sucesso!


KeyError: 'metodo_pagamento'

In [31]:
import pandas as pd

df = pd.read_csv('Atendimento_Sinistros_Tratado_Final.csv')

df = df.drop_duplicates()

df.to_csv('Atendimento_Sinistros_Tratado_Final.csv', index=False)

In [32]:
len(df)

93132

In [2]:
import pandas as pd

df_final_tratada = pd.read_csv('Base_Unificada_Outer.csv')

# Garantir tipos corretos (caso necessário)
cols = ['tipo_cobertura_premium', 'tipo_cobertura_basica', 'tipo_cobertura_padrao']

for col in cols:
    df_final_tratada[col] = df_final_tratada[col].astype(float)

# Criar colunas de número de apólices por tipo
df_final_tratada['num_apolices_premium'] = (
    df_final_tratada['num_apolices_ativas'] * df_final_tratada['tipo_cobertura_premium']
)

df_final_tratada['num_apolices_basica'] = (
    df_final_tratada['num_apolices_ativas'] * df_final_tratada['tipo_cobertura_basica']
)

df_final_tratada['num_apolices_padrao'] = (
    df_final_tratada['num_apolices_ativas'] * df_final_tratada['tipo_cobertura_padrao']
)

df_final_tratada.to_csv('Base_Unificada_Outer.csv', index=False)

print("Colunas criadas com sucesso!")

display(df_final_tratada[
    [
        'cod_individuo',
        'num_apolices_ativas',
        'tipo_cobertura_premium',
        'tipo_cobertura_basica',
        'tipo_cobertura_padrao',
        'num_apolices_premium',
        'num_apolices_basica',
        'num_apolices_padrao'
    ]
].head())

Colunas criadas com sucesso!


,cod_individuo,num_apolices_ativas,tipo_cobertura_premium,tipo_cobertura_basica,tipo_cobertura_padrao,num_apolices_premium,num_apolices_basica,num_apolices_padrao
0,221300000040,2.0,0.0,0.0,1.0,0.0,0.0,2.0
1,221300000051,1.0,0.0,0.0,1.0,0.0,0.0,1.0
2,221300000085,3.0,0.0,1.0,0.0,0.0,3.0,0.0
3,221300000119,2.0,0.0,0.0,1.0,0.0,0.0,2.0
4,221300000218,NaN,0.0,0.0,1.0,NaN,NaN,NaN


In [3]:
import pandas as pd

df_final_tratada = pd.read_csv('Base_Unificada_Tratada.csv')

# Garantir tipos corretos (caso necessário)
cols = ['tipo_cobertura_premium', 'tipo_cobertura_basica', 'tipo_cobertura_padrao']

for col in cols:
    df_final_tratada[col] = df_final_tratada[col].astype(float)

# Criar colunas de número de apólices por tipo
df_final_tratada['num_apolices_premium'] = (
    df_final_tratada['num_apolices_ativas'] * df_final_tratada['tipo_cobertura_premium']
)

df_final_tratada['num_apolices_basica'] = (
    df_final_tratada['num_apolices_ativas'] * df_final_tratada['tipo_cobertura_basica']
)

df_final_tratada['num_apolices_padrao'] = (
    df_final_tratada['num_apolices_ativas'] * df_final_tratada['tipo_cobertura_padrao']
)

df_final_tratada.to_csv('Base_Unificada_Tratada.csv', index=False)

print("Colunas criadas com sucesso!")

display(df_final_tratada[
    [
        'cod_individuo',
        'num_apolices_ativas',
        'tipo_cobertura_premium',
        'tipo_cobertura_basica',
        'tipo_cobertura_padrao',
        'num_apolices_premium',
        'num_apolices_basica',
        'num_apolices_padrao'
    ]
].head())

Colunas criadas com sucesso!


,cod_individuo,num_apolices_ativas,tipo_cobertura_premium,tipo_cobertura_basica,tipo_cobertura_padrao,num_apolices_premium,num_apolices_basica,num_apolices_padrao
0,221300904264,2.0,1.0,0.0,0.0,2.0,0.0,0.0
1,221300318278,3.0,0.0,1.0,0.0,0.0,3.0,0.0
2,221302854940,NaN,1.0,0.0,0.0,NaN,NaN,NaN
3,221300164895,1.0,1.0,0.0,0.0,1.0,0.0,0.0
4,221302543275,1.0,0.0,1.0,0.0,0.0,1.0,0.0
